# NYC Taxi Demand Prediction - EDA & Feature Engineering## ObiettivoAnalizzare i 4 dataset TLC (Yellow, Green, FHV, High-Volume FHV) di Gennaio 2025 per costruire un dataset aggregato con un **indice di disponibilita** come target per il modello predittivo.## Approccio- **Granularita**: fasce di 30 minuti (48 fasce/giorno)- **Target**: `availability_index` = trip_count(zona, ora, giorno) / max_trip_count(zona)- **Classi**: 5 livelli da "Molto Difficile" a "Molto Facile"- **Unione**: tutti e 4 i dataset concatenati verticalmente## Dataset (Gennaio 2025)- `yellow_tripdata_2025-01.parquet` - Yellow Cabs- `green_tripdata_2025-01.parquet` - Green Cabs (Boro Taxis)- `fhv_tripdata_2025-01.parquet` - For-Hire Vehicles- `fhvhv_tripdata_2025-01.parquet` - High-Volume FHV (Uber, Lyft, Via, Juno)

In [ ]:
import pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport seaborn as snsimport pyarrow.parquet as pqimport warningsimport osfrom pathlib import Pathwarnings.filterwarnings('ignore')sns.set_style('whitegrid')plt.rcParams['figure.figsize'] = (14, 6)plt.rcParams['font.size'] = 11DATA_DIR = Path(r'C:\Users\andre\Desktop\Progetto_Accenture\data')OUTPUT_DIR = Path(r'C:\Users\andre\Desktop\Progetto_Accenture\output')OUTPUT_DIR.mkdir(exist_ok=True)FILES = {    'yellow': DATA_DIR / 'yellow_tripdata_2025-01.parquet',    'green': DATA_DIR / 'green_tripdata_2025-01.parquet',    'fhv': DATA_DIR / 'fhv_tripdata_2025-01.parquet',    'fhvhv': DATA_DIR / 'fhvhv_tripdata_2025-01.parquet'}# Campionamento HVFHV per limiti di memoria# Impostare a 1.0 quando si ha piu RAM disponibileSAMPLE_FHVHV = 0.1print("Setup completato")

## 2. Taxi Zone LookupNYC e divisa in **265 zone taxi**. Ogni zona ha un ID, un nome, un borough e un tipo di servizio.

In [ ]:
zones = pd.read_csv('https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv')print(f"Zone totali: {len(zones)}")print(f"\n=== Per Borough ===")print(zones[zones['Borough'] != 'N/A']['Borough'].value_counts())print(f"\n=== Per Service Zone ===")print(zones['service_zone'].value_counts())zones.head(10)

## 3. Analisi Individuale dei Dataset### Come funzionano i taxi a NYC| Tipo | Descrizione | Area ||------|------------|------|| **Yellow Cab** | Taxi gialli | Ovunque (dominanti a Manhattan) || **Green Cab** | Taxi verdi | Outer boroughs + Upper Manhattan || **FHV** | For-Hire Vehicle | Tutta NYC || **HVFHV** | High-Volume FHV (Uber, Lyft) | Tutta NYC |> Yellow/Green: street-hail. FHV/HVFHV: prenotazione tramite app.

### 3.1 Yellow Taxi

In [ ]:
print("Caricamento Yellow Taxi...")yellow = pd.read_parquet(FILES['yellow'])print(f"Shape: {yellow.shape[0]:,} righe x {yellow.shape[1]} colonne")print(f"Memoria: {yellow.memory_usage(deep=True).sum() / 1024**2:.1f} MB")yellow.head(3)

In [ ]:
print("=== YELLOW - Schema ===")for col in yellow.columns:    dtype = str(yellow[col].dtype)    nulls = yellow[col].isna().sum()    null_pct = nulls / len(yellow) * 100    unique = yellow[col].nunique()    print(f"  {col:<30} | {dtype:<20} | nulls: {nulls:>8,} ({null_pct:>5.1f}%) | unique: {unique:>8,}")

In [ ]:
# Distribuzione temporale Yellowyellow['pickup_hour'] = pd.to_datetime(yellow['tpep_pickup_datetime']).dt.houryellow['pickup_dow'] = pd.to_datetime(yellow['tpep_pickup_datetime']).dt.dayofweekfig, axes = plt.subplots(1, 2, figsize=(16, 5))hourly = yellow['pickup_hour'].value_counts().sort_index()sns.barplot(x=hourly.index, y=hourly.values, ax=axes[0], color='#FFD700', edgecolor='black')axes[0].set_title('Yellow - Corse per Ora', fontsize=14, fontweight='bold')axes[0].set_xticks(range(24))dow_names = ['Lun', 'Mar', 'Mer', 'Gio', 'Ven', 'Sab', 'Dom']dow = yellow['pickup_dow'].value_counts().sort_index()dow.index = [dow_names[i] for i in dow.index]sns.barplot(x=dow.index, y=dow.values, ax=axes[1], color='#FFD700', edgecolor='black')axes[1].set_title('Yellow - Corse per Giorno', fontsize=14, fontweight='bold')plt.tight_layout()plt.savefig(OUTPUT_DIR / '01_yellow_temporal.png', dpi=150, bbox_inches='tight')plt.show()

### 3.2 Green Taxi

In [ ]:
print("Caricamento Green Taxi...")green = pd.read_parquet(FILES['green'])print(f"Shape: {green.shape[0]:,} righe x {green.shape[1]} colonne")green.head(3)

In [ ]:
green['pickup_hour'] = pd.to_datetime(green['lpep_pickup_datetime']).dt.hourgreen['pickup_dow'] = pd.to_datetime(green['lpep_pickup_datetime']).dt.dayofweekfig, axes = plt.subplots(1, 2, figsize=(16, 5))hourly_g = green['pickup_hour'].value_counts().sort_index()sns.barplot(x=hourly_g.index, y=hourly_g.values, ax=axes[0], color='#3CB371', edgecolor='black')axes[0].set_title('Green - Corse per Ora', fontsize=14, fontweight='bold')axes[0].set_xticks(range(24))dow_g = green['pickup_dow'].value_counts().sort_index()dow_g.index = [dow_names[i] for i in dow_g.index]sns.barplot(x=dow_g.index, y=dow_g.values, ax=axes[1], color='#3CB371', edgecolor='black')axes[1].set_title('Green - Corse per Giorno', fontsize=14, fontweight='bold')plt.tight_layout()plt.savefig(OUTPUT_DIR / '02_green_temporal.png', dpi=150, bbox_inches='tight')plt.show()print(f"\nGreen ha solo {len(green):,} corse - molto meno di Yellow.")

### 3.3 FHV (For-Hire Vehicle)

In [ ]:
print("Caricamento FHV...")fhv = pd.read_parquet(FILES['fhv'])print(f"Shape: {fhv.shape[0]:,} righe x {fhv.shape[1]} colonne")fhv.head(3)

In [ ]:
print("=== FHV - Schema ===")for col in fhv.columns:    dtype = str(fhv[col].dtype)    nulls = fhv[col].isna().sum()    null_pct = nulls / len(fhv) * 100    print(f"  {col:<30} | {dtype:<20} | nulls: {nulls:>8,} ({null_pct:>5.1f}%)")fhv_null_pct = fhv['PUlocationID'].isna().sum() / len(fhv) * 100print(f"\nPUlocationID ha il {fhv_null_pct:.1f}% di missing!")print(f"   Righe con location valida: {(~fhv['PUlocationID'].isna()).sum():,}")print(f"\nDecisione: Manteniamo SOLO righe con PUlocationID valido (1-265).")

In [ ]:
# FHV temporale (solo con location valida)fhv_valid = fhv[fhv['PUlocationID'].notna() & fhv['PUlocationID'].between(1, 265)].copy()fhv_valid['pickup_hour'] = pd.to_datetime(fhv_valid['pickup_datetime']).dt.hourfhv_valid['pickup_dow'] = pd.to_datetime(fhv_valid['pickup_datetime']).dt.dayofweekfig, axes = plt.subplots(1, 2, figsize=(16, 5))hourly_f = fhv_valid['pickup_hour'].value_counts().sort_index()sns.barplot(x=hourly_f.index, y=hourly_f.values, ax=axes[0], color='#FF6347', edgecolor='black')axes[0].set_title('FHV - Corse per Ora (con location)', fontsize=14, fontweight='bold')axes[0].set_xticks(range(24))dow_f = fhv_valid['pickup_dow'].value_counts().sort_index()dow_f.index = [dow_names[i] for i in dow_f.index]sns.barplot(x=dow_f.index, y=dow_f.values, ax=axes[1], color='#FF6347', edgecolor='black')axes[1].set_title('FHV - Corse per Giorno (con location)', fontsize=14, fontweight='bold')plt.tight_layout()plt.savefig(OUTPUT_DIR / '03_fhv_temporal.png', dpi=150, bbox_inches='tight')plt.show()

### 3.4 HVFHV (High-Volume FHV)

In [ ]:
print("Caricamento HVFHV (solo 5 colonne per memoria)...")# Carichiamo solo le colonne necessarie per evitare MemoryErrorfhvhv_cols = ['pickup_datetime', 'PULocationID', 'DOLocationID', 'trip_time', 'trip_miles']table = pq.read_table(FILES['fhvhv'], columns=fhvhv_cols)fhvhv = table.to_pandas()print(f"Colonne caricate: {fhvhv_cols}")print(f"Shape: {fhvhv.shape[0]:,} righe x {fhvhv.shape[1]} colonne")print(f"Memoria: {fhvhv.memory_usage(deep=True).sum() / 1024**2:.1f} MB")fhvhv.head(3)

In [ ]:
print("=== HVFHV - Schema (5 colonne selezionate) ===")for col in fhvhv.columns:    dtype = str(fhvhv[col].dtype)    nulls = fhvhv[col].isna().sum()    print(f"  {col:<25} | {dtype:<20} | nulls: {nulls:>8,}")print("\nNota: hvfhs_license_num non e stato caricato per risparmiare memoria.")print("   Per l'analisi per compagnia, ricaricare con quella colonna aggiunta.")

In [ ]:
# HVFHV temporalefhvhv['pickup_hour'] = pd.to_datetime(fhvhv['pickup_datetime']).dt.hourfhvhv['pickup_dow'] = pd.to_datetime(fhvhv['pickup_datetime']).dt.dayofweekfig, axes = plt.subplots(1, 2, figsize=(16, 5))hourly_h = fhvhv['pickup_hour'].value_counts().sort_index()sns.barplot(x=hourly_h.index, y=hourly_h.values, ax=axes[0], color='#8A2BE2', edgecolor='black')axes[0].set_title('HVFHV - Corse per Ora', fontsize=14, fontweight='bold')axes[0].set_xticks(range(24))dow_h = fhvhv['pickup_dow'].value_counts().sort_index()dow_h.index = [dow_names[i] for i in dow_h.index]sns.barplot(x=dow_h.index, y=dow_h.values, ax=axes[1], color='#8A2BE2', edgecolor='black')axes[1].set_title('HVFHV - Corse per Giorno', fontsize=14, fontweight='bold')plt.tight_layout()plt.savefig(OUTPUT_DIR / '04_fhvhv_temporal.png', dpi=150, bbox_inches='tight')plt.show()

In [ ]:
# HVFHV durata e distanzafhvhv['trip_duration_min'] = fhvhv['trip_time'] / 60fig, axes = plt.subplots(1, 2, figsize=(16, 5))dur = fhvhv[fhvhv['trip_duration_min'] <= 60]['trip_duration_min']sns.histplot(dur, bins=60, ax=axes[0], color='#8A2BE2', edgecolor='black')axes[0].set_title('HVFHV - Durata Corse (0-60 min)', fontsize=14, fontweight='bold')axes[0].axvline(dur.median(), color='red', linestyle='--', label=f'Mediana: {dur.median():.1f} min')axes[0].legend()miles = fhvhv[fhvhv['trip_miles'] <= 30]['trip_miles']sns.histplot(miles, bins=60, ax=axes[1], color='#8A2BE2', edgecolor='black')axes[1].set_title('HVFHV - Distanza Corse (0-30 mi)', fontsize=14, fontweight='bold')axes[1].axvline(miles.median(), color='red', linestyle='--', label=f'Mediana: {miles.median():.1f} mi')axes[1].legend()plt.tight_layout()plt.savefig(OUTPUT_DIR / '05_fhvhv_dur_dist.png', dpi=150, bbox_inches='tight')plt.show()

### 3.5 Riepilogo Comparativo

In [ ]:
summary = pd.DataFrame({    'Dataset': ['Yellow', 'Green', 'FHV', 'HVFHV'],    'Righe': [len(yellow), len(green), len(fhv), len(fhvhv)],    'Colonne': [len(yellow.columns), len(green.columns), len(fhv.columns), len(fhvhv.columns)],    'Location_NULL_pct': [        yellow['PULocationID'].isna().sum() / len(yellow) * 100,        green['PULocationID'].isna().sum() / len(green) * 100,        fhv['PUlocationID'].isna().sum() / len(fhv) * 100,        fhvhv['PULocationID'].isna().sum() / len(fhvhv) * 100    ]})print(summary.to_string(index=False))fig, axes = plt.subplots(1, 2, figsize=(16, 5))sns.barplot(data=summary, x='Dataset', y='Righe', ax=axes[0], palette=['#FFD700', '#3CB371', '#FF6347', '#8A2BE2'])axes[0].set_title('Volume Dati per Dataset', fontsize=14, fontweight='bold')for i, v in enumerate(summary['Righe']):    axes[0].text(i, v + 100000, f'{v:,}', ha='center', fontweight='bold', fontsize=9)sns.barplot(data=summary, x='Dataset', y='Location_NULL_pct', ax=axes[1], palette=['#FFD700', '#3CB371', '#FF6347', '#8A2BE2'])axes[1].set_title('% Location Mancanti', fontsize=14, fontweight='bold')for i, v in enumerate(summary['Location_NULL_pct']):    axes[1].text(i, v + 1, f'{v:.1f}%', ha='center', fontweight='bold')plt.tight_layout()plt.savefig(OUTPUT_DIR / '06_comparison.png', dpi=150, bbox_inches='tight')plt.show()

## 4. Pulizia e StandardizzazioneStandardizziamo i nomi delle colonne e filtriamo le righe invalide.

In [ ]:
def clean_dataset(df, dataset_type):    """Pulisce e standardizza un dataset. Restituisce DataFrame con colonne uniformi."""    df = df.copy()        if dataset_type == 'yellow':        df = df.rename(columns={            'tpep_pickup_datetime': 'pickup_datetime',            'tpep_dropoff_datetime': 'dropoff_datetime'        })    elif dataset_type == 'green':        df = df.rename(columns={            'lpep_pickup_datetime': 'pickup_datetime',            'lpep_dropoff_datetime': 'dropoff_datetime'        })    elif dataset_type == 'fhv':        df = df.rename(columns={            'PUlocationID': 'PULocationID',            'DOlocationID': 'DOLocationID',            'dropOff_datetime': 'dropoff_datetime'        })    # fhvhv already has correct names        df['taxi_type'] = dataset_type        # Filtra location valida    before = len(df)    df = df[df['PULocationID'].notna()]    df['PULocationID'] = df['PULocationID'].astype(int)    df = df[df['PULocationID'].between(1, 265)]        # Filtra durata valida    if 'dropoff_datetime' in df.columns:        df['trip_duration_sec'] = (            pd.to_datetime(df['dropoff_datetime']) - pd.to_datetime(df['pickup_datetime'])        ).dt.total_seconds()    elif 'trip_time' in df.columns:        df['trip_duration_sec'] = df['trip_time'].astype(float)        df = df[(df['trip_duration_sec'] > 0) & (df['trip_duration_sec'] < 86400)]        print(f"  {dataset_type:>6}: {before:>10,} -> {len(df):>10,} righe (rimosse {before - len(df):>8,})")        return df[['pickup_datetime', 'PULocationID', 'taxi_type', 'trip_duration_sec']].copy()print("Pulizia in corso...\n")yellow_c = clean_dataset(yellow, 'yellow')green_c = clean_dataset(green, 'green')fhv_c = clean_dataset(fhv, 'fhv')fhvhv_c = clean_dataset(fhvhv, 'fhvhv')total = len(yellow_c) + len(green_c) + len(fhv_c) + len(fhvhv_c)print(f"\nTOTALE: {total:,} righe pulite")

## 5. Feature EngineeringCreiamo feature temporali e geografiche comuni.

In [ ]:
def add_features(df):    """Aggiunge feature temporali e geografiche."""    df = df.copy()    df['pickup_datetime'] = pd.to_datetime(df['pickup_datetime'])        df['hour'] = df['pickup_datetime'].dt.hour    df['minute'] = df['pickup_datetime'].dt.minute    df['half_hour_bucket'] = df['hour'] * 2 + (df['minute'] // 30)    df['day_of_week'] = df['pickup_datetime'].dt.dayofweek    df['month'] = df['pickup_datetime'].dt.month    df['is_weekend'] = df['day_of_week'] >= 5    df['is_rush_hour'] = df['hour'].isin([7, 8, 9, 17, 18, 19])    df['is_night'] = (df['hour'] >= 22) | (df['hour'] < 5)    df['trip_duration_min'] = df['trip_duration_sec'] / 60        # JOIN con zone lookup    zone_info = zones.set_index('LocationID')    df = df.join(zone_info, on='PULocationID', how='left')    df = df.rename(columns={'Zone': 'zone_name'})        return dfprint("Feature engineering...")yellow_f = add_features(yellow_c)green_f = add_features(green_c)fhv_f = add_features(fhv_c)fhvhv_f = add_features(fhvhv_c)print(f"Completato. Colonne: {list(yellow_f.columns)}")

## 6. Unione dei DatasetConcateniamo verticalmente tutti i dataset.

In [ ]:
cols = ['pickup_datetime', 'PULocationID', 'taxi_type', 'trip_duration_sec',        'hour', 'minute', 'half_hour_bucket', 'day_of_week', 'month',        'is_weekend', 'is_rush_hour', 'is_night', 'trip_duration_min',        'borough', 'service_zone', 'zone_name']all_trips = pd.concat([    yellow_f[cols], green_f[cols], fhv_f[cols], fhvhv_f[cols]], ignore_index=True)print(f"Dataset unito: {len(all_trips):,} righe x {len(all_trips.columns)} colonne")print(f"\n=== Distribuzione per Tipo ===")for t, c in all_trips['taxi_type'].value_counts().items():    print(f"  {t:>8}: {c:>10,} ({c/len(all_trips)*100:.1f}%)")

In [ ]:
# Visualizzazione unionefig, axes = plt.subplots(1, 3, figsize=(18, 5))type_dist = all_trips['taxi_type'].value_counts()colors_map = {'yellow': '#FFD700', 'green': '#3CB371', 'fhv': '#FF6347', 'fhvhv': '#8A2BE2'}axes[0].pie(type_dist.values, labels=[t.upper() for t in type_dist.index],            autopct='%1.1f%%', colors=[colors_map[t] for t in type_dist.index])axes[0].set_title('Distribuzione per Tipo', fontsize=14, fontweight='bold')hourly_type = all_trips.groupby(['taxi_type', 'hour']).size().unstack(fill_value=0)hourly_type = hourly_type.div(hourly_type.sum(axis=1), axis=0) * 100hourly_type.T.plot(kind='bar', stacked=True, ax=axes[1], color=colors_map, width=0.8)axes[1].set_title('Mix Taxi per Ora (%)', fontsize=14, fontweight='bold')axes[1].set_xlabel('Ora')axes[1].set_ylabel('%')axes[1].legend(title='Tipo')borough_dist = all_trips[all_trips['borough'] != 'N/A']['borough'].value_counts()sns.barplot(x=borough_dist.index, y=borough_dist.values, ax=axes[2], palette='Set2')axes[2].set_title('Corse per Borough', fontsize=14, fontweight='bold')axes[2].tick_params(axis='x', rotation=45)plt.tight_layout()plt.savefig(OUTPUT_DIR / '07_combined.png', dpi=150, bbox_inches='tight')plt.show()

## 7. AggregazioneAggreghiamo per (zona, fascia 30min, giorno, mese).

In [ ]:
print("Aggregazione...")aggregated = all_trips.groupby(    ['PULocationID', 'half_hour_bucket', 'day_of_week', 'month']).agg(    trip_count=('pickup_datetime', 'count'),    unique_taxi_types=('taxi_type', 'nunique'),    avg_trip_duration_min=('trip_duration_min', 'mean'),    median_trip_duration_min=('trip_duration_min', 'median'),    std_trip_duration_min=('trip_duration_min', 'std'),).reset_index()zone_info = zones.set_index('LocationID')aggregated = aggregated.join(zone_info, on='PULocationID', how='left')aggregated = aggregated.rename(columns={'Zone': 'zone_name'})aggregated['is_weekend'] = aggregated['day_of_week'] >= 5aggregated['is_rush_hour'] = aggregated['half_hour_bucket'].apply(lambda b: (b // 2) in [7, 8, 9, 17, 18, 19])aggregated['is_night'] = aggregated['half_hour_bucket'].apply(lambda b: (b // 2) >= 22 or (b // 2) < 5)total_possible = 265 * 48 * 7 * 1coverage = len(aggregated) / total_possible * 100print(f"Aggregato: {len(aggregated):,} combinazioni")print(f"   Copertura: {coverage:.1f}% ({total_possible - len(aggregated):,} combinazioni vuote)")

## 8. Target: Availability Index`availability_index` = trip_count(zona, ora, giorno) / max_trip_count(zona)Normalizzato per zona per rendere le classi comparabili.

In [ ]:
max_per_zone = aggregated.groupby('PULocationID')['trip_count'].max().rename('max_trip_count_zone')aggregated = aggregated.join(max_per_zone, on='PULocationID')aggregated['availability_index'] = aggregated['trip_count'] / aggregated['max_trip_count_zone']bins = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]labels = ['Molto Difficile', 'Difficile', 'Medio', 'Facile', 'Molto Facile']aggregated['availability_class'] = pd.cut(    aggregated['availability_index'], bins=bins, labels=labels, include_lowest=True)class_mapping = {label: i for i, label in enumerate(labels)}aggregated['availability_class_id'] = aggregated['availability_class'].map(class_mapping)print("Target creato!\n")print("=== Distribuzione Classi ===")for cls, count in aggregated['availability_class'].value_counts().sort_index().items():    pct = count / len(aggregated) * 100    print(f"  {cls:>15}: {count:>6,} ({pct:>5.1f}%)")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))sns.histplot(aggregated['availability_index'], bins=50, ax=axes[0], color='#4CAF50', edgecolor='black')axes[0].set_title('Distribuzione Availability Index', fontsize=14, fontweight='bold')for b in [0.2, 0.4, 0.6, 0.8]:    axes[0].axvline(b, color='gray', linestyle=':', alpha=0.7)class_counts = aggregated['availability_class'].value_counts().sort_index()class_colors = ['#D32F2F', '#FF9800', '#FFC107', '#8BC34A', '#4CAF50']sns.barplot(x=class_counts.index, y=class_counts.values, ax=axes[1], palette=class_colors)axes[1].set_title('Distribuzione Classi', fontsize=14, fontweight='bold')axes[1].tick_params(axis='x', rotation=30)axes[2].pie(class_counts.values, labels=class_counts.index, autopct='%1.1f%%', colors=class_colors)axes[2].set_title('Proporzione Classi', fontsize=14, fontweight='bold')plt.tight_layout()plt.savefig(OUTPUT_DIR / '08_availability.png', dpi=150, bbox_inches='tight')plt.show()

## 9. EDA sul Dataset Aggregato

In [ ]:
# Pattern orarihourly_avail = aggregated.groupby('half_hour_bucket').agg(    avg_avail=('availability_index', 'mean'),    total_trips=('trip_count', 'sum')).reset_index()fig, axes = plt.subplots(1, 2, figsize=(16, 5))sns.lineplot(data=hourly_avail, x='half_hour_bucket', y='avg_avail', ax=axes[0], color='#4CAF50', marker='o', markersize=4)axes[0].set_title('Disponibilita Media per Fascia Oraria', fontsize=14, fontweight='bold')axes[0].set_xticks(range(0, 48, 4))axes[0].set_xticklabels([f"{i//2:02d}:{'00' if i%2==0 else '30'}" for i in range(0, 48, 4)], rotation=45)sns.lineplot(data=hourly_avail, x='half_hour_bucket', y='total_trips', ax=axes[1], color='#2196F3', marker='o', markersize=4)axes[1].set_title('Trip Count Totale per Fascia', fontsize=14, fontweight='bold')axes[1].set_xticks(range(0, 48, 4))axes[1].set_xticklabels([f"{i//2:02d}:{'00' if i%2==0 else '30'}" for i in range(0, 48, 4)], rotation=45)plt.tight_layout()plt.savefig(OUTPUT_DIR / '09_hourly_patterns.png', dpi=150, bbox_inches='tight')plt.show()

In [ ]:
# Pattern per giornodow_names_full = ['Lunedi', 'Martedi', 'Mercoledi', 'Giovedi', 'Venerdi', 'Sabato', 'Domenica']dow_avail = aggregated.groupby('day_of_week').agg(    avg_avail=('availability_index', 'mean'),    total_trips=('trip_count', 'sum')).reset_index()dow_avail['day_name'] = dow_avail['day_of_week'].map(dict(enumerate(dow_names_full)))fig, axes = plt.subplots(1, 2, figsize=(16, 5))sns.barplot(data=dow_avail, x='day_name', y='avg_avail', ax=axes[0], palette='Greens')axes[0].set_title('Disponibilita per Giorno', fontsize=14, fontweight='bold')axes[0].tick_params(axis='x', rotation=45)sns.barplot(data=dow_avail, x='day_name', y='total_trips', ax=axes[1], palette='Blues')axes[1].set_title('Trip Count per Giorno', fontsize=14, fontweight='bold')axes[1].tick_params(axis='x', rotation=45)plt.tight_layout()plt.savefig(OUTPUT_DIR / '10_daily_patterns.png', dpi=150, bbox_inches='tight')plt.show()

In [ ]:
# Heatmap: Giorno x Oradow_hour = aggregated.groupby(['day_of_week', 'half_hour_bucket'])['availability_index'].mean().unstack(fill_value=0)dow_hour.index = [dow_names_full[i] for i in dow_hour.index]fig, ax = plt.subplots(figsize=(16, 6))sns.heatmap(dow_hour, cmap='RdYlGn', ax=ax, vmin=0, vmax=1, cbar_kws={'label': 'Availability Index'})ax.set_title('Disponibilita: Giorno x Fascia Oraria', fontsize=14, fontweight='bold')ax.set_xticks(range(0, 48, 4))ax.set_xticklabels([f"{i//2:02d}:{'00' if i%2==0 else '30'}" for i in range(0, 48, 4)], rotation=45)plt.tight_layout()plt.savefig(OUTPUT_DIR / '11_heatmap_dow_hour.png', dpi=150, bbox_inches='tight')plt.show()

In [ ]:
# Correlazionicorr_data = aggregated[['PULocationID', 'half_hour_bucket', 'day_of_week', 'month',    'trip_count', 'unique_taxi_types', 'avg_trip_duration_min',    'availability_index', 'availability_class_id']].copy()corr_data['is_weekend'] = aggregated['is_weekend'].astype(int)corr_data['is_rush_hour'] = aggregated['is_rush_hour'].astype(int)corr_data['is_night'] = aggregated['is_night'].astype(int)fig, ax = plt.subplots(figsize=(10, 8))sns.heatmap(corr_data.corr(), annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=ax, square=True)ax.set_title('Matrice di Correlazione', fontsize=14, fontweight='bold')plt.tight_layout()plt.savefig(OUTPUT_DIR / '12_correlations.png', dpi=150, bbox_inches='tight')plt.show()print("\n=== Correlazioni con availability_index ===")for feat, corr in corr_data.corr()['availability_index'].sort_values(ascending=False).items():    if feat != 'availability_index':        print(f"  {feat:>30}: {corr:+.4f}")

## 10. Salvataggio Dataset Finale

In [ ]:
model_features = [    'PULocationID', 'half_hour_bucket', 'day_of_week', 'month',    'unique_taxi_types', 'avg_trip_duration_min',    'is_weekend', 'is_rush_hour', 'is_night',    'borough', 'service_zone',    'availability_class_id', 'availability_index',]model_df = aggregated[model_features].copy()model_df['is_weekend'] = model_df['is_weekend'].astype(int)model_df['is_rush_hour'] = model_df['is_rush_hour'].astype(int)model_df['is_night'] = model_df['is_night'].astype(int)from sklearn.preprocessing import LabelEncoderle_borough = LabelEncoder()model_df['borough_encoded'] = le_borough.fit_transform(model_df['borough'].fillna('Unknown'))le_service = LabelEncoder()model_df['service_zone_encoded'] = le_service.fit_transform(model_df['service_zone'].fillna('Unknown'))aggregated.to_csv(OUTPUT_DIR / 'aggregated_dataset_jan2025.csv', index=False)aggregated.to_parquet(OUTPUT_DIR / 'aggregated_dataset_jan2025.parquet', index=False)model_df.to_csv(OUTPUT_DIR / 'model_ready_jan2025.csv', index=False)model_df.to_parquet(OUTPUT_DIR / 'model_ready_jan2025.parquet', index=False)import joblibjoblib.dump(le_borough, OUTPUT_DIR / 'le_borough.pkl')joblib.dump(le_service, OUTPUT_DIR / 'le_service.pkl')print("Dataset salvati nella cartella output/")print(f"\nRIEPILOGO:")print(f"   Dataset grezzo totale: {len(yellow)+len(green)+len(fhv)+len(fhvhv):,} righe")print(f"   Dopo pulizia: {len(all_trips):,} righe")print(f"   Dopo aggregazione: {len(aggregated):,} combinazioni")print(f"   Feature modello: {len(model_df.columns)}")print(f"\nPronto per il training del modello!")

## 11. Funzione Riutilizzabile per Mesi Successivi

In [ ]:
def process_month(yellow_path, green_path, fhv_path, fhvhv_path, month_label=""):    """Processa un mese di dati TLC e restituisce dataset aggregato e pronto per il modello."""    print(f"\n{'='*60}")    print(f"Processing: {month_label}")    print(f"{'='*60}")        y = pd.read_parquet(yellow_path)    g = pd.read_parquet(green_path)    f = pd.read_parquet(fhv_path)    cols_h = ['pickup_datetime', 'PULocationID', 'DOLocationID', 'trip_time', 'trip_miles']    h = pq.read_table(fhvhv_path, columns=cols_h).to_pandas()        y_c = clean_dataset(y, 'yellow')    g_c = clean_dataset(g, 'green')    f_c = clean_dataset(f, 'fhv')    h_c = clean_dataset(h, 'fhvhv')        y_f = add_features(y_c)    g_f = add_features(g_c)    f_f = add_features(f_c)    h_f = add_features(h_c)        cols = ['pickup_datetime', 'PULocationID', 'taxi_type', 'trip_duration_sec',            'hour', 'minute', 'half_hour_bucket', 'day_of_week', 'month',            'is_weekend', 'is_rush_hour', 'is_night', 'trip_duration_min',            'borough', 'service_zone', 'zone_name']        all_t = pd.concat([y_f[cols], g_f[cols], f_f[cols], h_f[cols]], ignore_index=True)        agg = all_t.groupby(['PULocationID', 'half_hour_bucket', 'day_of_week', 'month']).agg(        trip_count=('pickup_datetime', 'count'),        unique_taxi_types=('taxi_type', 'nunique'),        avg_trip_duration_min=('trip_duration_min', 'mean'),        median_trip_duration_min=('trip_duration_min', 'median'),        std_trip_duration_min=('trip_duration_min', 'std'),    ).reset_index()        zone_info = zones.set_index('LocationID')    agg = agg.join(zone_info, on='PULocationID', how='left')    agg = agg.rename(columns={'Zone': 'zone_name'})    agg['is_weekend'] = agg['day_of_week'] >= 5    agg['is_rush_hour'] = agg['half_hour_bucket'].apply(lambda b: (b // 2) in [7, 8, 9, 17, 18, 19])    agg['is_night'] = agg['half_hour_bucket'].apply(lambda b: (b // 2) >= 22 or (b // 2) < 5)        max_z = agg.groupby('PULocationID')['trip_count'].max().rename('max_trip_count_zone')    agg = agg.join(max_z, on='PULocationID')    agg['availability_index'] = agg['trip_count'] / agg['max_trip_count_zone']    agg['availability_class'] = pd.cut(        agg['availability_index'], bins=[0, 0.2, 0.4, 0.6, 0.8, 1.0],        labels=['Molto Difficile', 'Difficile', 'Medio', 'Facile', 'Molto Facile'],        include_lowest=True    )    agg['availability_class_id'] = agg['availability_class'].map(class_mapping)        mdf = agg[['PULocationID', 'half_hour_bucket', 'day_of_week', 'month',        'unique_taxi_types', 'avg_trip_duration_min', 'is_weekend', 'is_rush_hour', 'is_night',        'borough', 'service_zone', 'availability_class_id', 'availability_index']].copy()    mdf['is_weekend'] = mdf['is_weekend'].astype(int)    mdf['is_rush_hour'] = mdf['is_rush_hour'].astype(int)    mdf['is_night'] = mdf['is_night'].astype(int)    mdf['borough_encoded'] = le_borough.transform(mdf['borough'].fillna('Unknown'))    mdf['service_zone_encoded'] = le_service.transform(mdf['service_zone'].fillna('Unknown'))        print(f"{month_label}: {len(agg):,} combinazioni")    return agg, mdfprint("Funzione process_month() definita.")